# Session 6: MCP Deep Dive + Agentic Integration
**Duration:** 1.5 Hours | **Day 2** | **Hands-On Workshop**

---

## Learning Objectives
By the end of this session, participants will:
1. Build a fully functional MCP client that connects to MCP servers
2. Connect LangChain agents to MCP tools
3. Run MCP servers inside Docker containers
4. Understand production patterns for MCP deployment
5. Build an end-to-end agent that uses MCP for tool access

---

## Prerequisites
- Completed Session 4 (MCP intro) and Session 5 (Agents)
- MCP server from Session 4: `mcp_college_server.py`
- Docker installed and running
- Packages: `pip install mcp langchain langchain-ollama`

---
## Part 1: MCP Client -- Connecting to Servers (Theory + Hands-On - 25 min)

### 1.1 Recap: MCP Architecture

```
+-------------------+          +-------------------+
|   YOUR APP (Host) |          |   MCP SERVER      |
|                   |          |                   |
| +---------------+ |  stdio   | +---------------+ |
| | MCP Client    |<--------->| | Tools         | |
| +---------------+ |  or SSE  | | Resources     | |
|                   |          | | Prompts       | |
| +---------------+ |          | +---------------+ |
| | LLM / Agent   | |          |                   |
| +---------------+ |          |                   |
+-------------------+          +-------------------+
```

### 1.2 MCP Client Responsibilities
1. **Connect** to MCP server (stdio or SSE)
2. **Discover** available tools, resources, prompts
3. **Call tools** with arguments and receive results
4. **Read resources** (data sources)
5. **Use prompts** (pre-built templates)
6. **Handle errors** gracefully

In [ ]:
# 1.3 Building an MCP Client
# The MCP client connects to a server process and communicates via JSON-RPC

# First, let's create the MCP client code as a reusable module

mcp_client_code = '''
# mcp_client.py
# A reusable MCP client that connects to any MCP server

import asyncio
import json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


class MCPClient:
    """A client that connects to an MCP server and exposes its tools."""

    def __init__(self):
        self.session = None
        self.tools = []
        self.resources = []
        self._read = None
        self._write = None

    async def connect(self, command: str, args: list = None, env: dict = None):
        """Connect to an MCP server."""
        server_params = StdioServerParameters(
            command=command,
            args=args or [],
            env=env
        )

        # Create stdio transport
        self._stdio_context = stdio_client(server_params)
        streams = await self._stdio_context.__aenter__()
        self._read, self._write = streams

        # Create session
        self._session_context = ClientSession(self._read, self._write)
        self.session = await self._session_context.__aenter__()

        # Initialize
        await self.session.initialize()

        # Discover tools
        tools_response = await self.session.list_tools()
        self.tools = tools_response.tools

        print(f"Connected! Found {len(self.tools)} tools:")
        for tool in self.tools:
            print(f"  - {tool.name}: {tool.description[:60]}...")

    async def call_tool(self, tool_name: str, arguments: dict) -> str:
        """Call a tool on the MCP server."""
        result = await self.session.call_tool(tool_name, arguments)
        return result.content[0].text if result.content else "No result"

    async def list_resources(self):
        """List available resources."""
        resources = await self.session.list_resources()
        return resources.resources

    async def read_resource(self, uri: str) -> str:
        """Read a resource by URI."""
        result = await self.session.read_resource(uri)
        return result.contents[0].text if result.contents else "No content"

    async def disconnect(self):
        """Disconnect from the server."""
        if self._session_context:
            await self._session_context.__aexit__(None, None, None)
        if self._stdio_context:
            await self._stdio_context.__aexit__(None, None, None)

    def get_tools_schema(self):
        """Get tools in a format suitable for LLM function calling."""
        schemas = []
        for tool in self.tools:
            schemas.append({
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.inputSchema
            })
        return schemas


async def main():
    """Example usage of the MCP client."""
    client = MCPClient()

    try:
        # Connect to the college server
        await client.connect("python", ["mcp_college_server.py"])

        # Call some tools
        print("\n=== Calculator ===")
        result = await client.call_tool("calculator", {
            "operation": "multiply",
            "a": 25,
            "b": 4
        })
        print(result)

        print("\n=== Fee Calculator ===")
        result = await client.call_tool("calculate_total_fees", {
            "quota": "government",
            "years": 4,
            "include_hostel": True,
            "include_transport": False
        })
        print(result)

        print("\n=== GPA Calculator ===")
        result = await client.call_tool("calculate_gpa", {
            "grades": "S,A,B,A,S,C,A"
        })
        print(result)

        # Read resources
        print("\n=== College Info Resource ===")
        resources = await client.list_resources()
        for r in resources:
            print(f"  Resource: {r.uri}")
            content = await client.read_resource(str(r.uri))
            print(f"  Content: {content[:200]}...")

    finally:
        await client.disconnect()
        print("\nDisconnected.")


if __name__ == "__main__":
    asyncio.run(main())
'''

# Save the client
with open("../data/mcp_client.py", "w") as f:
    f.write(mcp_client_code)

print("MCP client code saved to ../data/mcp_client.py")
print("\nThis client can:")
print("  1. Connect to any MCP server (stdio)")
print("  2. Discover and call tools")
print("  3. List and read resources")
print("  4. Export tool schemas for LLM integration")

In [ ]:
# 1.4 Understanding the MCP Client Code

# Key concepts:

print("""
=== MCP Client Key Concepts ===

1. CONNECTION:
   - StdioServerParameters defines how to start the server
   - stdio_client creates the transport (stdin/stdout pipes)
   - ClientSession manages the protocol

2. DISCOVERY:
   - session.list_tools() -- discover available tools
   - session.list_resources() -- discover available data
   - session.list_prompts() -- discover available templates

3. TOOL CALLING:
   - session.call_tool(name, arguments) -- execute a tool
   - Returns structured result with content

4. RESOURCE READING:
   - session.read_resource(uri) -- read data
   - URI-based access (e.g., "college://info")

5. LIFECYCLE:
   - connect() -- start server, create session
   - ... use tools and resources ...
   - disconnect() -- clean shutdown
""")

---
## Part 2: Connecting LangChain Agent to MCP (Hands-On - 25 min)

The real power comes when we connect LLM agents to MCP tools. The agent can discover tools at runtime and use them for reasoning.

In [ ]:
# 2.1 Bridge: Convert MCP tools to LangChain tools
# This allows LangChain agents to use MCP tools seamlessly

agent_mcp_bridge_code = '''
# mcp_langchain_bridge.py
# Bridge that connects MCP tools to LangChain agents

import asyncio
import json
from typing import Any
from langchain_core.tools import StructuredTool
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


class MCPLangChainBridge:
    """Bridge between MCP servers and LangChain agents."""

    def __init__(self):
        self.session = None
        self.mcp_tools = []
        self.langchain_tools = []
        self._stdio_context = None
        self._session_context = None

    async def connect(self, command: str, args: list = None):
        """Connect to MCP server and convert tools to LangChain format."""
        server_params = StdioServerParameters(
            command=command,
            args=args or []
        )

        self._stdio_context = stdio_client(server_params)
        streams = await self._stdio_context.__aenter__()
        read, write = streams

        self._session_context = ClientSession(read, write)
        self.session = await self._session_context.__aenter__()
        await self.session.initialize()

        # Get MCP tools
        tools_response = await self.session.list_tools()
        self.mcp_tools = tools_response.tools

        # Convert to LangChain tools
        self.langchain_tools = []
        for mcp_tool in self.mcp_tools:
            lc_tool = self._create_langchain_tool(mcp_tool)
            self.langchain_tools.append(lc_tool)

        return self.langchain_tools

    def _create_langchain_tool(self, mcp_tool):
        """Convert an MCP tool to a LangChain StructuredTool."""
        session = self.session
        tool_name = mcp_tool.name

        def tool_func(**kwargs) -> str:
            """Synchronous wrapper for async MCP tool call."""
            loop = asyncio.get_event_loop()
            if loop.is_running():
                import concurrent.futures
                with concurrent.futures.ThreadPoolExecutor() as pool:
                    result = pool.submit(
                        asyncio.run,
                        session.call_tool(tool_name, kwargs)
                    ).result()
            else:
                result = asyncio.run(session.call_tool(tool_name, kwargs))

            return result.content[0].text if result.content else "No result"

        return StructuredTool(
            name=mcp_tool.name,
            description=mcp_tool.description or "",
            func=tool_func,
            args_schema=mcp_tool.inputSchema
        )

    async def disconnect(self):
        """Clean up connections."""
        if self._session_context:
            await self._session_context.__aexit__(None, None, None)
        if self._stdio_context:
            await self._stdio_context.__aexit__(None, None, None)
'''

with open("../data/mcp_langchain_bridge.py", "w") as f:
    f.write(agent_mcp_bridge_code)

print("MCP-LangChain bridge saved!")
print("\nThis bridge:")
print("  1. Connects to any MCP server")
print("  2. Discovers MCP tools")
print("  3. Converts them to LangChain StructuredTool objects")
print("  4. Agent can use them as if they were native LangChain tools")

In [ ]:
# 2.2 Simulating the MCP-to-LangChain flow
# Since MCP requires subprocess communication, we'll demonstrate the pattern
# by creating equivalent LangChain tools that mirror our MCP server

from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
import json

# These tools mirror the MCP server tools -- in production, the bridge
# automatically converts MCP tools to these LangChain tools

@tool
def mcp_calculator(operation: str, a: float, b: float) -> str:
    """[MCP Tool] Perform arithmetic operations: add, subtract, multiply, divide.

    Args:
        operation: One of 'add', 'subtract', 'multiply', 'divide'
        a: First number
        b: Second number
    """
    ops = {"add": a+b, "subtract": a-b, "multiply": a*b, "divide": a/b if b else "err"}
    return f"{a} {operation} {b} = {ops.get(operation, 'unknown op')}"

@tool
def mcp_fee_calculator(quota: str, years: int, include_hostel: bool, include_transport: bool) -> str:
    """[MCP Tool] Calculate total college fees at SIT.

    Args:
        quota: 'government', 'comedk', or 'management'
        years: Number of years (1-4)
        include_hostel: Include hostel fees (Rs 75,000/year)
        include_transport: Include transport fees (Rs 15,000/year)
    """
    fees = {"government": 55000, "comedk": 120000, "management": 250000}
    tuition = fees.get(quota, 0)
    hostel = 75000 if include_hostel else 0
    transport = 15000 if include_transport else 0
    total_per_year = tuition + hostel + transport
    return json.dumps({"quota": quota, "per_year": total_per_year, "total": total_per_year * years}, indent=2)

@tool
def mcp_gpa_calculator(grades: str) -> str:
    """[MCP Tool] Calculate GPA from VTU grades. Grade scale: S=10, A=9, B=8, C=7, D=6, E=5, F=0.

    Args:
        grades: Comma-separated grades like 'S,A,B,A,S,C'
    """
    gp = {"S":10,"A":9,"B":8,"C":7,"D":6,"E":5,"F":0}
    gl = [g.strip().upper() for g in grades.split(",")]
    pts = [gp[g] for g in gl if g in gp]
    gpa = sum(pts)/len(pts) if pts else 0
    return json.dumps({"grades": gl, "gpa": round(gpa,2)}, indent=2)

@tool
def mcp_weather(city: str) -> str:
    """[MCP Tool] Get weather for a city (mock data).

    Args:
        city: Name of the city
    """
    data = {"tumkur": 28, "bangalore": 24, "mysore": 26}
    temp = data.get(city.lower(), 25)
    return f"Weather in {city}: {temp}C, Partly Cloudy"


# Create the MCP-powered agent
mcp_tools = [mcp_calculator, mcp_fee_calculator, mcp_gpa_calculator, mcp_weather]
llm = ChatOllama(model="llama3.2", temperature=0)
llm_with_mcp = llm.bind_tools(mcp_tools)

print("MCP-powered agent ready with tools:")
for t in mcp_tools:
    print(f"  - {t.name}")

In [ ]:
# 2.3 Run the MCP-powered agent

def run_mcp_agent(query: str):
    """Run agent that uses MCP tools."""
    print(f"\nQuery: {query}")
    print("-" * 50)

    tool_map = {t.name: t for t in mcp_tools}

    messages = [
        SystemMessage(content="""You are a college assistant for SIT, Tumkur.
        You have access to MCP tools for calculations, fees, GPA, and weather.
        Always use tools for accurate information. Be helpful and concise."""),
        HumanMessage(content=query)
    ]

    for _ in range(5):
        response = llm_with_mcp.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            print(f"\nAnswer: {response.content}")
            return response.content

        for tc in response.tool_calls:
            print(f"  [MCP Call] {tc['name']}({tc['args']})")
            result = tool_map[tc['name']].invoke(tc['args'])
            print(f"  [Result] {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

    return "Max iterations reached"

# Test queries
run_mcp_agent("What is the total fee for 4 years COMEDK with hostel and transport?")

In [ ]:
run_mcp_agent("Calculate GPA for grades: S, A, B, A, A, S, B, C and tell me the classification")

In [ ]:
run_mcp_agent("What's the weather in Tumkur and Bangalore? Which city is warmer?")

---
## Part 3: MCP Servers in Docker (Hands-On - 20 min)

### 3.1 Why Docker for MCP?

- **Isolation**: Each MCP server runs in its own container
- **Portability**: Same server runs anywhere Docker runs
- **Security**: Container sandboxing limits what the server can access
- **Deployment**: Easy scaling and orchestration
- **Dependencies**: No Python version conflicts

### 3.2 Creating a Dockerized MCP Server

In [ ]:
# 3.2 Create Dockerfile for the MCP server

dockerfile_content = """# Dockerfile for MCP College Server
FROM python:3.11-slim

WORKDIR /app

# Install MCP SDK
RUN pip install --no-cache-dir mcp

# Copy the MCP server
COPY mcp_college_server.py .

# The server communicates via stdio
CMD ["python", "mcp_college_server.py"]
"""

with open("../data/Dockerfile.mcp", "w") as f:
    f.write(dockerfile_content)

print("Dockerfile created at ../data/Dockerfile.mcp")
print()
print("To build and use:")
print("  docker build -f Dockerfile.mcp -t mcp-college-server .")
print("  docker run -i mcp-college-server")
print()
print("For MCP config with Docker:")

docker_mcp_config = {
    "mcpServers": {
        "college-assistant": {
            "command": "docker",
            "args": ["run", "-i", "--rm", "mcp-college-server"]
        }
    }
}
print(json.dumps(docker_mcp_config, indent=2))

In [ ]:
# 3.3 Docker Compose for multiple MCP servers

docker_compose_content = """# docker-compose.yml
# Run multiple MCP servers together

version: '3.8'

services:
  # MCP College Server
  mcp-college:
    build:
      context: .
      dockerfile: Dockerfile.mcp
    stdin_open: true    # Required for stdio transport
    tty: true
    restart: unless-stopped

  # You can add more MCP servers here:
  # mcp-database:
  #   image: mcp/postgres-server
  #   environment:
  #     - DATABASE_URL=postgresql://...

  # mcp-filesystem:
  #   image: mcp/filesystem-server
  #   volumes:
  #     - ./data:/data:ro
"""

with open("../data/docker-compose.yml", "w") as f:
    f.write(docker_compose_content)

print("docker-compose.yml created!")
print()
print("Commands:")
print("  docker-compose build        # Build all servers")
print("  docker-compose up -d        # Start all servers")
print("  docker-compose logs -f      # View logs")
print("  docker-compose down         # Stop all servers")

---
## Part 4: MCP in Production -- Patterns and Best Practices (Theory - 10 min)

### 4.1 Production MCP Architecture

```
+-------------------------------------------+
|            YOUR APPLICATION                |
|                                           |
|  +-------------+    +------------------+  |
|  | LLM Agent   |--->| MCP Client Pool  |  |
|  +-------------+    +--+----+----+-----+  |
+----------------------|----|----|----------+
                       |    |    |
           +-----------+    |    +-----------+
           |                |                |
    +------v------+  +-----v------+  +------v------+
    | MCP Server  |  | MCP Server |  | MCP Server  |
    | (Database)  |  | (File Sys) |  | (API)       |
    | [Docker]    |  | [Docker]   |  | [Docker]    |
    +-------------+  +------------+  +-------------+
```

### 4.2 Best Practices

| Practice | Description |
|----------|------------|
| **Single Responsibility** | One MCP server per domain (DB, files, API) |
| **Error Handling** | Always return meaningful error messages |
| **Input Validation** | Validate all tool arguments before processing |
| **Logging** | Log all tool calls for debugging and auditing |
| **Docker** | Run each server in its own container |
| **Security** | Limit file system access, use read-only mounts |
| **Timeouts** | Set timeouts for long-running tools |
| **Documentation** | Clear docstrings -- the LLM reads them! |

### 4.3 Popular Pre-built MCP Servers

| Server | What It Does | Install |
|--------|-------------|--------|
| **filesystem** | Read/write files | `@modelcontextprotocol/server-filesystem` |
| **postgres** | Query PostgreSQL databases | `@modelcontextprotocol/server-postgres` |
| **github** | GitHub API (repos, issues, PRs) | `@modelcontextprotocol/server-github` |
| **slack** | Send/read Slack messages | `@modelcontextprotocol/server-slack` |
| **puppeteer** | Browser automation | `@modelcontextprotocol/server-puppeteer` |
| **sqlite** | SQLite database operations | `@modelcontextprotocol/server-sqlite` |

---
## Exercises

### Exercise 1: Extend the MCP Server
Add a new tool to the MCP server that checks scholarship eligibility based on CGPA and category (General/SC/ST/OBC).

### Exercise 2: Docker Build
Build and run the MCP server in Docker. Test it using the MCP Inspector:
```bash
docker build -f Dockerfile.mcp -t mcp-college-server .
npx @modelcontextprotocol/inspector docker run -i mcp-college-server
```

### Exercise 3: Multi-Server Agent
Create a second MCP server (e.g., for a library system) and build an agent that connects to both servers simultaneously.

---

## Summary

### Key Takeaways:
1. **MCP Clients** connect to servers, discover tools, and execute them
2. **MCP-LangChain Bridge** converts MCP tools into LangChain StructuredTools
3. **Agents + MCP** = LLM reasoning with standardized tool access
4. **Docker** provides isolation, portability, and security for MCP servers
5. **Production patterns**: Use one server per domain, containerize everything
6. MCP ecosystem has **pre-built servers** for databases, file systems, APIs, and more

### Next Session: Multi-Agent Systems
We will build systems where multiple agents collaborate to solve complex tasks.